# Campaign ROI

Translating the A/B test result into a dollar estimate. Since cost and revenue per conversion aren't in the dataset, this picks reasonable placeholder numbers and ends with a sensitivity table so the ROI conclusion doesn't rest on one guess.

In [1]:
import pandas as pd

### assumptions (not in the dataset, just guesses)

Cost per ad impression and revenue per conversion aren't provided, these are placeholder numbers to make the ROI concrete. Swap in real numbers if available, the sensitivity table at the end shows how much the conclusion depends on them.

In [2]:
COST_PER_AD_IMPRESSION = 0.02   # usd per ad shown
REVENUE_PER_CONVERSION = 25.00  # usd per conversion

Loading the raw data.

In [3]:
df = pd.read_csv("../data/raw/marketing_AB.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df.head()

,user id,test group,converted,total ads,most ads day,most ads hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14


### cost

Campaign cost, using total ad impressions shown to the ad group only, the psa group is the control and wasn't a paid campaign.

In [4]:
ad_group = df[df["test group"] == "ad"]
psa_group = df[df["test group"] == "psa"]

total_ad_impressions = ad_group["total ads"].sum()
campaign_cost = total_ad_impressions * COST_PER_AD_IMPRESSION

print(f"total ad impressions: {total_ad_impressions:,}")
print(f"campaign cost: ${campaign_cost:,.2f}")

total ad impressions: 14,014,701
campaign cost: $280,294.02


In [5]:
top_1pct_cutoff = ad_group["total ads"].quantile(0.99)
top_1pct_share = ad_group.loc[ad_group["total ads"] >= top_1pct_cutoff, "total ads"].sum() / total_ad_impressions
print(f"top 1% of ad users (>= {top_1pct_cutoff:.0f} ads) = {top_1pct_share:.1%} of total impressions")

top 1% of ad users (>= 201 ads) = 13.2% of total impressions


### revenue

Incremental revenue: actual conversions in the ad group, minus what would have converted at the psa group's (lower) conversion rate. That difference is the revenue attributable to the ad campaign, not just the raw conversions.

In [6]:
n_ad = len(ad_group)
ad_conversion_rate = ad_group["converted"].mean()
psa_conversion_rate = psa_group["converted"].mean()

actual_conversions = ad_group["converted"].sum()
actual_revenue = actual_conversions * REVENUE_PER_CONVERSION

counterfactual_conversions = n_ad * psa_conversion_rate
counterfactual_revenue = counterfactual_conversions * REVENUE_PER_CONVERSION

incremental_revenue = actual_revenue - counterfactual_revenue

print(f"ad group: {n_ad:,} users, {ad_conversion_rate:.4%} conversion")
print(f"psa group: {psa_conversion_rate:.4%} conversion")
print(f"actual conversions: {actual_conversions:,.0f} -> ${actual_revenue:,.2f}")
print(f"counterfactual conversions: {counterfactual_conversions:,.1f} -> ${counterfactual_revenue:,.2f}")
print(f"incremental revenue: ${incremental_revenue:,.2f}")

ad group: 564,577 users, 2.5547% conversion
psa group: 1.7854% conversion
actual conversions: 14,423 -> $360,575.00
counterfactual conversions: 10,080.0 -> $252,000.45
incremental revenue: $108,574.55


### roi

Incremental revenue divided by campaign cost.

In [7]:
roi = incremental_revenue / campaign_cost if campaign_cost > 0 else float("nan")
print(f"campaign cost: ${campaign_cost:,.2f}")
print(f"incremental revenue: ${incremental_revenue:,.2f}")
print(f"roi: {roi:.2f}x")

campaign cost: $280,294.02
incremental revenue: $108,574.55
roi: 0.39x


### sensitivity

Since both dollar assumptions were guesses, checking how ROI changes across a range of costs and revenue per conversion, rather than trusting a single number.

In [8]:
cost_scenarios = [0.01, 0.02, 0.05, 0.10]
revenue_scenarios = [10, 25, 50, 100]

rows = []
for cost in cost_scenarios:
    for revenue in revenue_scenarios:
        c = total_ad_impressions * cost
        inc_rev = actual_conversions * revenue - counterfactual_conversions * revenue
        rows.append({
            "cost_per_impression": cost,
            "revenue_per_conversion": revenue,
            "campaign_cost": round(c, 2),
            "incremental_revenue": round(inc_rev, 2),
            "roi_x": round(inc_rev / c, 2) if c > 0 else None,
        })

pd.DataFrame(rows)

,cost_per_impression,revenue_per_conversion,campaign_cost,incremental_revenue,roi_x
0,0.01,10,140147.01,43429.82,0.31
1,0.01,25,140147.01,108574.55,0.77
2,0.01,50,140147.01,217149.11,1.55
3,0.01,100,140147.01,434298.21,3.10
4,0.02,10,280294.02,43429.82,0.15
5,0.02,25,280294.02,108574.55,0.39
6,0.02,50,280294.02,217149.11,0.77
7,0.02,100,280294.02,434298.21,1.55
8,0.05,10,700735.05,43429.82,0.06
9,0.05,25,700735.05,108574.55,0.15
